# LLM inference benchmark for Medical RAG — Colab runner

Runs experiments **E1–E4** of [`docs/llm_benchmark_spec.md`](../../docs/llm_benchmark_spec.md) on a Colab **L4** GPU
(24 GB): Qwen2.5-7B-Instruct served by **vLLM** (FP16 and AWQ), **Ollama** (GGUF Q4_K_M) and plain
**HF Transformers**, replaying the real prompts in `prompts/medical_rag_prompts.jsonl`.

**Before you start**
1. Push the repository to GitHub — the notebook clones it, and the prompts file must be in the commit.
2. *Runtime → Change runtime type →* **L4 GPU**.
3. *Secrets* (key icon in the left bar), with notebook access enabled:
   `OPENAI_API_KEY` (answer-quality judge for E3); optional `HF_TOKEN`; `GITHUB_TOKEN` only if the repository is private.
4. Keep this tab open while experiments run (Colab Pro stops when the tab closes; background execution is a Pro+ feature).

**Time budget on L4** (each phase is capped by `max_duration_s` in its config):

| Step | Configurations | Estimate |
| --- | --- | --- |
| Setup (install + ~25 GB of weights) | | 20–30 min |
| E1 — one user | vLLM FP16, vLLM AWQ, Ollama, HF | 2–2.5 h |
| E2 — 1→32 users | vLLM FP16, vLLM AWQ, Ollama, HF | 5–5.5 h |
| E3 — 16 users + quality | vLLM FP16, vLLM AWQ, Ollama | ~30 min + judge |
| E4 — prefix caching | vLLM FP16 off / on | ~1 h |

About 9–10 GPU hours in total. Check the compute-unit rate of the L4 runtime in Colab's *Resources* panel.

**Resuming.** Every configuration is copied to Google Drive when it ends. Re-running a cell skips configurations
already finished on Drive, so the experiments can be spread over several sessions. A configuration interrupted
half-way restarts from the beginning; its partial copy is ignored by the analysis.

E5 (TGI) needs Docker, which Colab does not provide, so it is not run here.

In [ ]:
REPO_URL = "https://github.com/chaubaokhanh12/LLM-Inference-Benchmarking-for-Medical-RAG.git"
BRANCH = "main"
REPO_DIR = "/content/repo"
DRIVE_RESULTS = "/content/drive/MyDrive/llm-inference-bench/results"
VLLM_SPEC = "vllm==0.31.0"            # wheels need an NVIDIA driver with CUDA >= 13 (checked below)
JUDGE_MODEL = "gpt-4.1-mini-2025-04-14"  # same judge as the medical-rag evaluation

## 1. GPU and driver

In [ ]:
import re
import subprocess

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
print(smi)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert "L4" in gpu, f"Expected an L4 runtime, got {gpu!r}: Runtime -> Change runtime type -> L4 GPU"
m = re.search(r"CUDA (?:UMD )?Version:\s*([\d.]+)", smi)
driver_cuda = float(m.group(1)) if m else 0.0
print(f"GPU: {gpu} | driver supports CUDA {driver_cuda}")
if driver_cuda < 13:
    print("WARNING: vLLM 0.31 wheels are built for CUDA 13 and will not start on this driver. "
          "Set VLLM_SPEC to an older vLLM release built for CUDA 12, re-run the first cell, then continue.")

## 2. Google Drive and secrets

In [ ]:
import os

from google.colab import drive, userdata

drive.mount("/content/drive")
os.makedirs(DRIVE_RESULTS, exist_ok=True)

for name in ("OPENAI_API_KEY", "HF_TOKEN", "GITHUB_TOKEN"):
    try:
        value = userdata.get(name)
    except Exception:  # secret not defined, or notebook access not granted
        value = None
    if value:
        os.environ[name] = value
os.environ["EVAL_LLM_MODEL"] = JUDGE_MODEL
print({name: bool(os.environ.get(name)) for name in ("OPENAI_API_KEY", "HF_TOKEN", "GITHUB_TOKEN")})

## 3. Code, libraries and Ollama

In [ ]:
import os
import subprocess

url = REPO_URL
if os.environ.get("GITHUB_TOKEN"):
    url = url.replace("https://", f"https://x-access-token:{os.environ['GITHUB_TOKEN']}@")
if os.path.isdir(REPO_DIR):
    result = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True, text=True)
else:
    result = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, REPO_DIR],
                            capture_output=True, text=True)
if result.returncode != 0:  # message printed without the URL, so a token never shows up
    raise RuntimeError("git failed: " + result.stderr.replace(url, REPO_URL)[-500:])
os.chdir(f"{REPO_DIR}/inference-bench")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
assert os.path.exists("../prompts/medical_rag_prompts.jsonl"), "prompts/medical_rag_prompts.jsonl is not in the repository"

In [ ]:
!pip install -q "{VLLM_SPEC}"
!pip install -q -e ".[hf]"
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama_install.log 2>&1; tail -n 2 /tmp/ollama_install.log
!python -c "import vllm, torch, transformers; print('vllm', vllm.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__, '| CUDA', torch.cuda.is_available())"
!ollama --version

Pre-download the weights once per session, so server start-up time is not counted in the experiments
(FP16 ≈ 15 GB, AWQ ≈ 5.5 GB, GGUF ≈ 4.7 GB).

In [ ]:
import subprocess
import time
import urllib.request

from huggingface_hub import snapshot_download

for repo in ("Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-7B-Instruct-AWQ"):
    path = snapshot_download(repo, allow_patterns=["*.json", "*.safetensors", "*.txt", "tokenizer*"])
    print(repo, "->", path)

ollama = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2)
        break
    except Exception:
        time.sleep(1)
!ollama pull qwen2.5:7b-instruct-q4_K_M
ollama.terminate()
ollama.wait()
!df -h /content | tail -n 1

In [ ]:
!cat ../prompts/stats.json

## 4. Experiments

`run(name)` runs `python -m bench.run configs/<name>.yaml` (which starts the server, warms it up, measures every
concurrency level and repeat while sampling `nvidia-smi` and vLLM `/metrics`, then stops the server) and copies the
result folder to Drive. Configurations already finished on Drive are skipped.

In [ ]:
import glob
import os
import shutil

import yaml
from IPython import get_ipython


def _suffix(name):
    cfg = yaml.safe_load(open(f"configs/{name}.yaml", encoding="utf-8"))
    return f"_{cfg['experiment']}_{cfg['config_name']}"


def finished_on_drive(name):
    return sorted(d for d in glob.glob(f"{DRIVE_RESULTS}/*{_suffix(name)}") if os.path.exists(f"{d}/DONE"))


def run(name):
    done = finished_on_drive(name)
    if done:
        print(f"skip {name}: finished on Drive ({os.path.basename(done[-1])})")
        return done[-1]
    before = set(glob.glob(f"results/*{_suffix(name)}"))
    get_ipython().system(f"python -m bench.run configs/{name}.yaml")
    new = sorted(set(glob.glob(f"results/*{_suffix(name)}")) - before)
    if not new:
        raise RuntimeError(f"{name}: no results folder was created, see the output above")
    dest = f"{DRIVE_RESULTS}/{os.path.basename(new[-1])}"
    shutil.copytree(new[-1], dest, dirs_exist_ok=True)
    state = "finished" if os.path.exists(f"{dest}/DONE") else "INCOMPLETE, re-run this cell"
    print(f"{name}: {state} -> {dest}")
    return dest

### E1 — one user: vLLM vs Ollama vs HF Transformers (≈2–2.5 h)

In [ ]:
for name in ["e1_vllm_fp16", "e1_vllm_awq", "e1_ollama", "e1_hf"]:
    run(name)

### E2 — 1, 4, 8, 16, 32 concurrent users (≈5–5.5 h)

HF Transformers serves one request at a time, so at high concurrency its requests simply queue:
time to first token grows with the number of users while throughput stays flat. That is the expected result.

In [ ]:
for name in ["e2_vllm_fp16", "e2_vllm_awq", "e2_ollama", "e2_hf"]:
    run(name)

### E3 — FP16 vs AWQ (and Ollama) at 16 users, with answer quality (≈30 min + judge)

The same 100 prompts are answered by each configuration and scored by one LLM judge
(faithfulness to the passages in the prompt, correctness against the reference answer).

In [ ]:
for name in ["e3_vllm_fp16", "e3_vllm_awq", "e3_ollama"]:
    run(name)

In [ ]:
assert os.environ.get("OPENAI_API_KEY"), "Add OPENAI_API_KEY in Colab Secrets and re-run section 2"
fp16, awq, oll = (finished_on_drive(n)[-1] for n in ("e3_vllm_fp16", "e3_vllm_awq", "e3_ollama"))
!python -m bench.quality_eval "{fp16}" "{awq}" --n 100
!python -m bench.quality_eval "{oll}" --n 100

### E4 — prefix caching off vs on, 16 users (≈1 h)

The prefix cache is reset before every repeat, so only the shared system prompt can be reused, as in real traffic.

In [ ]:
for name in ["e4_vllm_prefix_off", "e4_vllm_prefix_on"]:
    run(name)

## 5. Summary and figures

In [ ]:
!python -m analysis.aggregate --results "{DRIVE_RESULTS}"
!python -m analysis.plots --results "{DRIVE_RESULTS}"

from IPython.display import Image, display

for png in sorted(glob.glob(f"{DRIVE_RESULTS}/plots/*.png")):
    print(os.path.basename(png))
    display(Image(png))

## 6. Bring the results back to the repository

Download the Drive folder `llm-inference-bench/results/` (or the zip below) into `inference-bench/results/` in your
local clone and commit it. Per-second samples (`gpu.csv`, `vllm_metrics.csv`) are git-ignored because of their
size; `summary.csv`, the plots, `requests.csv`, `env.json` and `command.txt` are what the README cites.

In [ ]:
!cd "{DRIVE_RESULTS}/.." && zip -qr results.zip results && ls -lh "{DRIVE_RESULTS}/../results.zip"